# Clase 009 — Excepciones y context managers

**Parte 0** · Python Tutorial cap. 8 + Ramalho cap. 18.

> 🎯 Manejo riguroso de errores y garantía de cleanup con `with`.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import os, time, tempfile
from contextlib import contextmanager
from pathlib import Path

## 1️⃣ Jerarquía de excepciones

```
BaseException
├── SystemExit          ← sys.exit()
├── KeyboardInterrupt   ← Ctrl+C
└── Exception           ← captura esto, no BaseException
    ├── ArithmeticError
    │   └── ZeroDivisionError
    ├── LookupError
    │   ├── KeyError
    │   └── IndexError
    ├── ValueError
    ├── TypeError
    ├── OSError
    │   └── FileNotFoundError
    └── ...
```

Captura `Exception` o más específico. **NUNCA** captures `BaseException` o uses `except:` solo.

## 2️⃣ `try/except/else/finally`

```python
try:
    valor = riesgoso()
except ValueError as e:
    log(f'valor inválido: {e}')
    valor = None
else:
    # Solo si try NO lanzó excepción
    log('OK')
finally:
    # Siempre, lanzó o no
    cleanup()
```

- `try` — código que puede fallar
- `except` — manejo específico
- `else` — éxito (raro, pero útil)
- `finally` — cleanup garantizado (lo que hace `with` automático)

In [ ]:
def parse_int_safe(s, default=0):
    """Convierte a int; default si no es parseable. Otros errores propagan."""
    try:
        return int(s)
    except ValueError:
        return default

print(parse_int_safe('42'))          # 42
print(parse_int_safe('foo'))         # 0
print(parse_int_safe('3.14'))        # 0
try:
    parse_int_safe({'a': 1})         # TypeError no es ValueError → propaga
except TypeError as e:
    print(f'TypeError correcto: {e}')

## 3️⃣ Capturar específico — por qué

```python
# ❌ TRAMPA: esconde TODO error, hasta tipo y nombre
try:
    valor = parse(linea)
except:
    valor = None   # bug silencioso

# ✅ CORRECTO: solo el error que sabes manejar
try:
    valor = parse(linea)
except ValueError as e:
    log(f'línea inválida {idx}: {e}')
    valor = None
```

Un `except:` puede ocultar un `KeyboardInterrupt`, un `NameError` (typo) o un `MemoryError`. Casi nunca es lo que quieres.

## 4️⃣ Excepciones propias

Las excepciones son **comunicación tipada**. En vez de:

```python
raise Exception('CSV corrupto en línea 42')
```

Define tu tipo:

```python
class DatasetCorruptoError(Exception):
    def __init__(self, mensaje, linea):
        super().__init__(mensaje)
        self.linea = linea

try:
    cargar(path)
except DatasetCorruptoError as e:
    log(f'línea {e.linea}: {e}')   # ahora caller puede ACTUAR
```

In [ ]:
class DatasetCorruptoError(Exception):
    def __init__(self, mensaje, linea):
        super().__init__(mensaje)
        self.linea = linea

def cargar_csv_estricto(lineas, n_cols):
    for i, linea in enumerate(lineas, start=1):
        cols = linea.split(',')
        if len(cols) != n_cols:
            raise DatasetCorruptoError(f'esperaba {n_cols} cols, vino {len(cols)}', linea=i)
        yield cols

datos = ['a,b,c', 'd,e,f', 'g,h']   # última línea corrupta
try:
    list(cargar_csv_estricto(datos, n_cols=3))
except DatasetCorruptoError as e:
    print(f'Error línea {e.linea}: {e}')

## 5️⃣ Context managers — `with`

```python
# Sin with: si parse() falla, el archivo queda abierto
f = open('data.csv')
datos = parse(f)
f.close()

# Con with: cleanup garantizado, incluso si parse() lanza
with open('data.csv') as f:
    datos = parse(f)
# aquí f ya está cerrado
```

Protocolo: el objeto debe tener `__enter__` (entrada) y `__exit__` (salida). `__exit__` recibe info de la excepción si la hubo.

In [ ]:
# Demo: with garantiza close incluso con excepción
tmp = Path(tempfile.mkdtemp()) / 'demo.txt'
tmp.write_text('linea1\nlinea2\nlinea3\n')

with open(tmp) as f:
    for linea in f:
        print(linea.strip())
print('archivo cerrado:', f.closed)

## 6️⃣ Context manager propio con `@contextmanager`

La forma corta: una función con `yield`. Antes del yield = `__enter__`. Después = `__exit__`.

```python
from contextlib import contextmanager

@contextmanager
def timer(label):
    t0 = time.perf_counter()
    yield                       # aquí corre el código del `with`
    dt = time.perf_counter() - t0
    print(f'{label}: {dt*1000:.1f} ms')

with timer('carga'):
    time.sleep(0.1)
```

In [ ]:
@contextmanager
def timer(label):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        dt = time.perf_counter() - t0
        print(f'{label}: {dt*1000:.1f} ms')

with timer('operación A'):
    time.sleep(0.05)

with timer('operación B'):
    sum(i*i for i in range(100_000))

In [ ]:
# Context manager práctico: cambiar de directorio temporalmente
@contextmanager
def cd(path):
    prev = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(prev)   # garantizado incluso si hay excepción

print('antes:', Path.cwd().name)
with cd(tempfile.gettempdir()):
    print('dentro:', Path.cwd().name)
print('después:', Path.cwd().name)

## ✅ Checklist

- [ ] Capturo excepciones específicas, no `except:`
- [ ] Sé crear una excepción propia con atributos
- [ ] Uso `with` para archivos y otros recursos
- [ ] Sé escribir un context manager con `@contextmanager`
- [ ] Entiendo que `finally` garantiza cleanup

## 📝 Homework

Ver `README.md`. `parse_int_safe`, `DatasetCorruptoError`, `timer`, `cd` context manager.

## 📖 Definiciones y características

**Excepción**

Objeto que se 'lanza' (`raise`) cuando algo anómalo ocurre. Sube por el stack hasta que un `except` lo captura, o termina el programa. Todas heredan de `BaseException`; las que debes capturar heredan de `Exception`.

**`try/except/else/finally`**

**`try`**: código riesgoso. **`except`**: maneja una excepción específica. **`else`**: corre si `try` no lanzó (raro pero útil). **`finally`**: cleanup garantizado, lanzó o no.

**Jerarquía de excepciones**

`BaseException` → `SystemExit` / `KeyboardInterrupt` / `Exception` → `ValueError` / `TypeError` / `LookupError` (→ `KeyError`, `IndexError`) / `OSError` / `ArithmeticError` (→ `ZeroDivisionError`). Captura siempre la más específica que sepas manejar.

**Excepción propia (custom)**

Subclase de `Exception` (o subclase específica). Permite tipificar errores de tu dominio (`class DatasetCorruptoError(Exception): ...`) en vez de strings. El caller puede `except DatasetCorruptoError` con precisión.

**Context manager**

Objeto con `__enter__` y `__exit__` que se usa con `with`. Garantiza setup/teardown (abrir/cerrar archivo, conectar/desconectar BD, lock/unlock). El `__exit__` corre incluso si hay excepción.

**`@contextmanager`**

Decorador de `contextlib` que convierte una función con `yield` en context manager. Pre-yield = `__enter__`, post-yield = `__exit__`. Mucho más corto que escribir la clase completa.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `except:` o `except Exception:` ciego esconde bugs | Cualquier error queda silenciado (incluso typos `NameError`). **Fix**: captura el tipo específico (`except ValueError as e`). Si quieres loguear cualquiera, `except Exception as e: log.error(...); raise`. |
| `finally` con `return` traga la excepción | Si `finally` ejecuta `return`, la excepción del `try` desaparece silenciosamente. **Fix**: NO uses `return` en `finally`; sólo cleanup. |
| `with open(f) as f, open(g) as g:` falla por sintaxis vieja | Esa sintaxis requiere Python 3.10+. **Fix**: en versiones viejas usa `contextlib.ExitStack` o `with open(f) as f: with open(g) as g:` anidado. |
| Capturé `KeyError` pero el código sigue rompiéndose | Tu `except` está más arriba del `try`, o el error proviene de otra línea. **Fix**: lee el traceback completo, busca el `KeyError` real y rodea solo esa línea con try. |
| Excepción dentro de generator no se captura como espero | Las excepciones en generators son tricky; `except` rodea el `for`, no el `yield`. **Fix**: lee la sección "Exceptions in generators" del PEP 380, o reformula como función normal. |

## ❓ Preguntas frecuentes

**❓ ¿`except Exception` o `except`?**

Casi siempre `except Exception` — más explícito. `except:` desnudo captura también `KeyboardInterrupt` y `SystemExit`, lo cual rompe Ctrl+C y `sys.exit()`.

**❓ ¿Cuándo creo una excepción propia?**

Cuando el caller necesita **diferenciar** este error de otros. `raise ValueError('CSV corrupto')` obliga al caller a parsear strings; `raise DatasetCorruptoError(linea=42)` le da un tipo y un atributo concreto.

**❓ ¿`with` solo para archivos?**

No — para CUALQUIER recurso que necesite cleanup. Files (`open`), conexiones DB (`engine.connect()`), locks (`threading.Lock`), sesiones HTTP (`requests.Session`), transacciones (`db.atomic()`), timing (`with timer(...)`).

**❓ ¿Debo capturar y re-lanzar para añadir contexto?**

Sí, con `raise ExceptionNueva(...) from e` — preserva el stacktrace original como '`__cause__`'. El log muestra ambos errores en cadena.

**❓ ¿`pass` en `except` es siempre malo?**

**Casi siempre sí.** Si tienes que silenciar, al menos `log.warning(...)`. Solo OK cuando la excepción es esperada (`except FileNotFoundError: pass` al limpiar archivos opcionales).

## 🔗 Referencias

- [Python Tutorial — Errors](https://docs.python.org/3/tutorial/errors.html)
- [contextlib](https://docs.python.org/3/library/contextlib.html)
- Ramalho, *Fluent Python* 2e, cap. 18

➡️ **Siguiente:** [010 — OOP básico, dataclasses, herencia](../010-oop-basico-dataclasses-herencia/README.md)

## ✅ Soluciones de los ejercicios

Intentá resolverlos vos primero; acá está una solución de referencia comentada. Todas las celdas corren sin internet ni archivos externos.

**Ejercicio 1.** `parse_int_safe(s, default=0)` que capture **solo** `ValueError` y demuestre que no esconde otros errores (ej. `TypeError` al pasar un dict).

In [ ]:
# Solución 1 — captura específica, no ciega
def parse_int_safe(s, default=0):
    """Convierte s a int; devuelve default si no es parseable.
    OJO: solo captura ValueError. Un TypeError (ej. dict) PROPAGA a propósito."""
    try:
        return int(s)
    except ValueError:
        return default

# Caso válido
assert parse_int_safe("42") == 42
# Caso inválido pero parseable-como-texto -> usa default
assert parse_int_safe("abc", default=-1) == -1
assert parse_int_safe("abc") == 0

# Caso "otro tipo": int({}) lanza TypeError, que NO capturamos -> se propaga
otro_error_propago = False
try:
    parse_int_safe({"no": "soy int"})
except TypeError:
    otro_error_propago = True  # bien: no lo escondimos

assert otro_error_propago, "El TypeError debería propagarse, no esconderse"
print("OK: ValueError -> default; TypeError se propaga como debe")

**Ejercicio 2.** `DatasetCorruptoError(Exception)` con atributo `linea`, lanzada desde `cargar_csv` cuando una fila no tiene el número correcto de columnas.

In [ ]:
# Solución 2 — excepción propia tipada con atributo de dominio
class DatasetCorruptoError(Exception):
    """Error de dominio: una fila del CSV no tiene el #columnas esperado."""
    def __init__(self, linea, mensaje="fila con número de columnas inválido"):
        self.linea = linea                     # atributo concreto, no un string
        super().__init__(f"{mensaje} (línea {linea})")

def cargar_csv(texto, n_columnas):
    """Parsea un CSV en memoria; valida que cada fila tenga n_columnas."""
    filas = []
    for i, linea in enumerate(texto.strip().splitlines(), start=1):
        campos = linea.split(",")
        if len(campos) != n_columnas:
            raise DatasetCorruptoError(linea=i)
        filas.append(campos)
    return filas

# CSV correcto
buen_csv = "a,b,c\n1,2,3\n4,5,6"
assert cargar_csv(buen_csv, n_columnas=3) == [["a","b","c"],["1","2","3"],["4","5","6"]]

# CSV corrupto: la línea 2 tiene solo 2 columnas
mal_csv = "a,b,c\n1,2\n4,5,6"
try:
    cargar_csv(mal_csv, n_columnas=3)
except DatasetCorruptoError as e:
    # el caller puede leer el atributo directamente, sin parsear texto
    assert e.linea == 2
    print(f"OK: detectado corrupto en línea {e.linea} -> {e}")

**Ejercicio 3.** Contar palabras leyendo un archivo con `with`, y comparar con la versión manual `open/close`: mostrar que si hay excepción a mitad, sin `with` el archivo queda abierto.

In [ ]:
# Solución 3 — `with` garantiza el cierre incluso si algo falla
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp()) / "texto.txt"
tmp.write_text("hola mundo\nadios mundo cruel\n", encoding="utf-8")

# Versión CORRECTA con with: cuenta palabras por línea
total = 0
with tmp.open(encoding="utf-8") as f:
    for linea in f:
        total += len(linea.split())
assert total == 5
print(f"Palabras contadas con `with`: {total}")

# Versión manual: si hay excepción a mitad, el close() NUNCA se ejecuta
f = tmp.open(encoding="utf-8")
quedo_abierto = False
try:
    next(f)                 # leo una línea
    raise RuntimeError("falla simulada a mitad de la lectura")
    f.close()               # <- esta línea nunca corre
except RuntimeError:
    quedo_abierto = not f.closed   # el archivo sigue abierto: recurso filtrado
finally:
    f.close()               # limpieza manual obligatoria

assert quedo_abierto is True
print("OK: sin `with`, tras la excepción el archivo quedó abierto (por eso usamos with)")

**Ejercicio 4.** Context manager `timer` con `@contextmanager`: `with timer("carga"):` imprime cuánto duró el bloque.

In [ ]:
# Solución 4 — timer como context manager con @contextmanager
import time
from contextlib import contextmanager

@contextmanager
def timer(label):
    t0 = time.perf_counter()          # pre-yield = __enter__
    try:
        yield                         # aquí corre el cuerpo del `with`
    finally:
        dt = time.perf_counter() - t0 # post-yield = __exit__ (corre siempre)
        print(f"[{label}] duró {dt:.4f} s")

with timer("carga"):
    total = sum(range(200_000))       # trabajo medible

# El finally corre incluso si el bloque lanza excepción:
midio_pese_al_error = False
try:
    with timer("bloque-con-error"):
        raise ValueError("boom")
except ValueError:
    midio_pese_al_error = True

assert midio_pese_al_error
print("OK: timer reporta segundos y mide aun cuando el bloque falla")

**Ejercicio 5.** Context manager `cd(path)`: cambia de directorio al entrar y vuelve al salir, **incluso si hay excepción**.

In [ ]:
# Solución 5 — cd: cambia de cwd y SIEMPRE lo restaura
import os
import tempfile
from pathlib import Path
from contextlib import contextmanager

@contextmanager
def cd(path):
    prev = Path.cwd()                 # guardo el cwd original
    os.chdir(path)                    # entro al nuevo directorio
    try:
        yield Path(path)
    finally:
        os.chdir(prev)                # vuelvo pase lo que pase

origen = Path.cwd()
destino = Path(tempfile.mkdtemp()).resolve()

# Caso normal
with cd(destino):
    assert Path.cwd().resolve() == destino
assert Path.cwd() == origen, "debe volver al cwd original al salir"

# Caso con excepción a mitad: igual restaura
try:
    with cd(destino):
        raise RuntimeError("falla dentro del bloque")
except RuntimeError:
    pass
assert Path.cwd() == origen, "debe restaurar el cwd aun con excepción"
print("OK: cd cambia y restaura el directorio incluso ante excepciones")